In [ ]:
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from allennlp.commands.elmo import ElmoEmbedder
from torch import nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset

# 加载 IMDb 数据集（用于情感分析）
dataset = load_dataset("imdb")

# 加载 ELMo 嵌入生成器
elmo = ElmoEmbedder()

# 定义一个简单的情感分析模型，使用 ELMo 嵌入和一个简单的全连接网络
class SentimentAnalysisModel(nn.Module):
    def __init__(self, hidden_size=128):
        super(SentimentAnalysisModel, self).__init__()
        self.fc1 = nn.Linear(1024, hidden_size)  # ELMo 输出维度为 1024
        self.fc2 = nn.Linear(hidden_size, 2)     # 输出情感标签数量（正面/负面）
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# 将文本转化为 ELMo 嵌入
def get_elmo_embeddings(texts):
    embeddings = []
    for text in texts:
        tokenized_text = text.split()  # 分词
        elmo_embedding = elmo.embed_sentence(tokenized_text)  # 获取 ELMo 嵌入
        embeddings.append(elmo_embedding.mean(axis=0))  # 对每个句子的嵌入求均值
    return torch.tensor(embeddings)

# 数据预处理：将 IMDB 数据集转化为 ELMo 嵌入和标签
def preprocess_data(dataset):
    texts = dataset['text']
    labels = dataset['label']
    embeddings = get_elmo_embeddings(texts)
    return embeddings, torch.tensor(labels)

# 加载训练和测试数据集
train_data, test_data = dataset['train'], dataset['test']

# 预处理数据
X_train, y_train = preprocess_data(train_data)
X_test, y_test = preprocess_data(test_data)

# 创建数据集和数据加载器
class SentimentDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = embeddings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]

train_dataset = SentimentDataset(X_train, y_train)
test_dataset = SentimentDataset(X_test, y_test)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64)

# 初始化模型
model = SentimentAnalysisModel(hidden_size=128)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

# 训练模型
num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    for batch in train_dataloader:
        embeddings, labels = batch
        optimizer.zero_grad()
        outputs = model(embeddings)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}/{num_epochs} completed.")

# 评估模型
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for batch in test_dataloader:
        embeddings, labels = batch
        outputs = model(embeddings)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

# 计算准确率
accuracy = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {accuracy:.4f}")